# Img2GPS — Walkthrough

CIS 5190 final project, Track A. We predict GPS coordinates from a single image taken on Penn's campus (test rectangle: 33rd & Walnut → 34th & Spruce). The official metric is the average Haversine distance in meters.

This notebook is the human-readable companion to the scripts:

- `Img2GPS/extract_exif.py` builds `metadata.csv`
- `Img2GPS/preprocess.py` provides `prepare_data` / `load_raw`
- `Img2GPS/model.py` defines the **MobileNetV3-Small + soft-cluster classifier** with hard-coded target stats and cluster-center buffers
- `Img2GPS/train.py` runs the training loop (K-means cluster head, CE + Haversine loss)
- `Img2GPS/eval_project_a.py` is the course-style evaluator

### Architecture rationale (why we moved off MSE regression)

The original ResNet-18 + 2-D MSE-regression head collapsed to predicting the training centroid (~80 m on the leaderboard, *worse* than the 48.7 m constant-mean baseline). The current setup predicts `K=16` softmax logits over location clusters and outputs the weighted average of the cluster centers in raw degrees — making mean-collapse architecturally impossible while still emitting `[lat, lon]` per the spec.


# Colab bootstrap
clone the repo and install dependencies on first run. Safe to re-run locally; it's a no-op when the repo already exists.

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/SheilaBkny/cis_5190_project.git"
REPO_BRANCH = "iter1"
COLAB_REPO_DIR = "/content/cis_5190_project"

IN_COLAB = "google.colab" in sys.modules

def _git(*args):
    subprocess.run(["git", "-C", COLAB_REPO_DIR, *args], check=True)

if IN_COLAB:
    if not os.path.exists(os.path.join(COLAB_REPO_DIR, "Img2GPS")):
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, COLAB_REPO_DIR],
            check=True,
        )
    else:
        # Repo dir already exists from an earlier session — pull the
        # latest commit on iter1 so new data / notebook fixes show up
        # without needing to delete /content/cis_5190_project manually.
        _git("fetch", "origin", REPO_BRANCH)
        _git("checkout", REPO_BRANCH)
        _git("reset", "--hard", f"origin/{REPO_BRANCH}")
    os.chdir(COLAB_REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )
    head = subprocess.check_output(["git", "-C", COLAB_REPO_DIR, "log", "-1", "--oneline"]).decode().strip()
    print(f"on commit: {head}")

print("cwd:", os.getcwd())

on commit: ccccada clustering
cwd: /content/cis_5190_project


In [2]:
import math
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while REPO_ROOT.name and not (REPO_ROOT / "Img2GPS").exists():
    REPO_ROOT = REPO_ROOT.parent
PROJECT_DIR = REPO_ROOT / "Img2GPS"
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(REPO_ROOT)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("count :", torch.cuda.device_count())
    print("mem GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print("torch :", torch.__version__)

from preprocess import load_raw, prepare_data  # noqa: E402
from model import Model  # noqa: E402
from train import haversine_meters, location_grouped_split  # noqa: E402

REPO_ROOT, PROJECT_DIR

cuda available: True
device: Tesla T4
count : 1
mem GB: 15.64
torch : 2.10.0+cu128


(PosixPath('/content/cis_5190_project'),
 PosixPath('/content/cis_5190_project/Img2GPS'))

## 1. Data summary

We collected 89 photos around Penn's campus and extracted GPS coordinates from EXIF + Apple location xattrs. The location-grouped split makes sure photos that share an exact GPS coordinate (about 8 photos per spot) live entirely on one side of the train/val split, which avoids leakage.

In [3]:
df = pd.read_csv(PROJECT_DIR / "metadata.csv")
print(f"rows: {len(df)}")
print(f"unique locations: {df[['latitude','longitude']].drop_duplicates().shape[0]}")
df.describe(percentiles=[0.05, 0.5, 0.95]).round(6)

rows: 89
unique locations: 60


,latitude,longitude
count,89.000000,89.000000
mean,39.951564,-75.191324
std,0.000253,0.000552
min,39.951063,-75.192170
5%,39.951187,-75.191850
50%,39.951497,-75.191612
95%,39.952113,-75.190313
max,39.952113,-75.190287


In [4]:
def haversine(a, b):
    R = 6_371_000.0
    lat1, lon1 = map(math.radians, a)
    lat2, lon2 = map(math.radians, b)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    h = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 2 * R * math.asin(math.sqrt(h))

mean_lat = df['latitude'].mean()
mean_lon = df['longitude'].mean()
lat_span = haversine((df['latitude'].min(), mean_lon), (df['latitude'].max(), mean_lon))
lon_span = haversine((mean_lat, df['longitude'].min()), (mean_lat, df['longitude'].max()))
print(f"bounding box: {lat_span:.1f} m (NS) x {lon_span:.1f} m (EW)")
print(f"training mean: ({mean_lat:.6f}, {mean_lon:.6f})")

constant_dist = [haversine((mean_lat, mean_lon), p) for p in df[['latitude','longitude']].values]
print(f"constant-mean baseline Haversine: mean={np.mean(constant_dist):.2f}m  p50={np.median(constant_dist):.2f}m  max={np.max(constant_dist):.2f}m")

bounding box: 116.8 m (NS) x 160.5 m (EW)
training mean: (39.951564, -75.191324)
constant-mean baseline Haversine: mean=48.72m  p50=42.60m  max=107.47m


In [ ]:
# Phone-captured images only. The test region is a tiny ~150 x 110 m
# rectangle on Penn's campus, and the spec sampling pattern (a few
# bearings per location) already gives in-domain coverage on this scale.
main_csv = PROJECT_DIR / "metadata.csv"
TRAIN_CSV = str(main_csv)
print(f"phone-only training: {len(pd.read_csv(main_csv))} rows  ({main_csv.name})")
TRAIN_CSV

## 2. Train

Trains on whatever `TRAIN_CSV` was set to in the previous cell — phone-only by default (`metadata.csv`, 89 rows).

Under the hood, `train.py` now:

1. Runs **K-means with K=16** on the training-split `(lat, lon)` to produce cluster centers.
2. Stamps the centers onto the model's `cluster_centers` buffer (saved inside `model.pt`).
3. Trains a **soft classifier**: backbone outputs K logits, `softmax(logits) @ centers` gives the predicted lat/lon in raw degrees.
4. Optimizes `cross_entropy(logits, closest_cluster) + 0.1 * (haversine_m / 1000)` so training is aligned with the eval metric.
5. Freezes everything except the final MobileNetV3-Small block + classifier head — essential with only ~70 training images.
6. Photometric-only augmentation (no horizontal flip, no rotation — those break bearing cues).

Saves the best-by-val-Haversine checkpoint to `Img2GPS/model.pt`. Skip this cell if you want to evaluate the existing checkpoint.

In [ ]:
# K=80 in model.py puts us in the instance-retrieval regime (~1-2 train
# images per cluster), so we lean more on the Haversine soft-pred loss
# (--haversine-loss-weight 0.5) since CE class signal becomes noisy.
print(f"TRAIN_CSV={TRAIN_CSV}")
!python Img2GPS/train.py --csv "{TRAIN_CSV}" --epochs 30 --lr 1e-3 --haversine-loss-weight 0.5

## 2.5 Save the trained model to your machine

`Img2GPS/model.pt` only exists inside the Colab VM after training — it is *not* automatically synced back to your laptop or to the `submission` branch. The next cell:

1. Prints the file's size and a short MD5 fingerprint so you can confirm it's the run you just produced.
2. If running in Colab, triggers a browser download.

After the download completes, place the file at `Img2GPS/model.pt` in your local clone and run `Img2GPS/scripts/promote_to_submission.sh` to copy it onto the `submission` branch and push — that's the version the leaderboard pulls.

In [7]:
import hashlib, os, sys

MODEL_PATH = str(PROJECT_DIR / "model.pt")
size_mb = os.path.getsize(MODEL_PATH) / 1e6
md5 = hashlib.md5(open(MODEL_PATH, "rb").read()).hexdigest()[:12]
print(f"model.pt   size: {size_mb:.1f} MB   md5: {md5}")

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(MODEL_PATH)
    print("Downloaded. Move ~/Downloads/model.pt -> <repo>/Img2GPS/model.pt locally,")
    print("then: bash Img2GPS/scripts/promote_to_submission.sh")
else:
    print("Not in Colab — model.pt is already on your local filesystem at the path above.")

model.pt   size: 6.5 MB   md5: c66ddba9814a


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded. Move ~/Downloads/model.pt -> <repo>/Img2GPS/model.pt locally,
then: bash Img2GPS/scripts/promote_to_submission.sh


## 3. Evaluate the saved checkpoint

We load `model.pt` (the best-by-Haversine snapshot) and report the official metrics on:

1. The full collected dataset.
2. The held-out validation split (location-grouped, the honest signal).
3. The course-provided reference set in `Img2GPS/reference/`.

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Model(weights_path=str(PROJECT_DIR / 'model.pt')).to(device).eval()
print(f"y_mean: {model.y_mean.tolist()}")
print(f"y_std:  {model.y_std.tolist()}")
print(f"K cluster centers (lat, lon):")
for i, (lat, lon) in enumerate(model.cluster_centers.tolist()):
    print(f"  [{i:02d}]  ({lat:.6f}, {lon:.6f})")

y_mean: [39.951541900634766, -75.19132232666016]
y_std:  [0.0002309196861460805, 0.0005374249303713441]
K cluster centers (lat, lon):
  [00]  (39.951187, -75.190620)
  [01]  (39.951698, -75.191757)
  [02]  (39.952522, -75.193245)
  [03]  (39.951698, -75.190399)
  [04]  (39.950809, -75.191086)
  [05]  (39.952084, -75.192795)
  [06]  (39.951180, -75.191689)
  [07]  (39.951405, -75.190437)
  [08]  (39.952110, -75.190315)
  [09]  (39.951481, -75.190796)
  [10]  (39.951401, -75.191254)
  [11]  (39.952621, -75.192917)
  [12]  (39.951485, -75.191803)
  [13]  (39.952461, -75.193504)
  [14]  (39.951111, -75.190773)
  [15]  (39.951553, -75.192169)
  [16]  (39.951309, -75.190491)
  [17]  (39.951492, -75.191605)
  [18]  (39.950935, -75.190926)
  [19]  (39.950706, -75.191208)
  [20]  (39.951073, -75.191475)
  [21]  (39.951580, -75.190422)
  [22]  (39.951275, -75.190521)
  [23]  (39.952553, -75.193130)
  [24]  (39.951466, -75.190422)
  [25]  (39.951603, -75.191734)
  [26]  (39.952663, -75.192780)
  

In [9]:
def evaluate(csv_path):
    X, y = prepare_data(str(csv_path))
    with torch.no_grad():
        preds = model(X.to(device)).cpu()
    distances = haversine_meters(preds, y).numpy()
    return preds, y, distances

preds_full, y_full, dists_full = evaluate(PROJECT_DIR / 'metadata.csv')
print(f"FULL set (n={len(y_full)}):")
print(f"  mean={dists_full.mean():.2f}m  p50={np.median(dists_full):.2f}m  p90={np.quantile(dists_full,0.9):.2f}m  max={dists_full.max():.2f}m")

FULL set (n=89):
  mean=40.41m  p50=33.72m  p90=76.09m  max=107.23m


In [10]:
_, y_all = load_raw(str(PROJECT_DIR / 'metadata.csv'))
train_idx, val_idx = location_grouped_split(y_all, val_fraction=0.2, seed=42)
X_full, _ = prepare_data(str(PROJECT_DIR / 'metadata.csv'))
with torch.no_grad():
    preds_val = model(X_full[val_idx].to(device)).cpu()
dists_val = haversine_meters(preds_val, y_all[val_idx]).numpy()
print(f"VAL set (held-out, n={len(val_idx)}):")
print(f"  mean={dists_val.mean():.2f}m  p50={np.median(dists_val):.2f}m  p90={np.quantile(dists_val,0.9):.2f}m  max={dists_val.max():.2f}m")

VAL set (held-out, n=18):
  mean=44.22m  p50=28.40m  p90=99.67m  max=106.60m


In [11]:
ref_csv = PROJECT_DIR / 'reference' / 'metadata.csv'
preds_ref, y_ref, dists_ref = evaluate(ref_csv)
print(f"REFERENCE set (n={len(y_ref)}):")
for (lat, lon), (plat, plon), d in zip(y_ref.tolist(), preds_ref.tolist(), dists_ref):
    print(f"  truth=({lat:.6f},{lon:.6f})  pred=({plat:.6f},{plon:.6f})  haversine={d:.2f}m")
print(f"reference mean Haversine: {dists_ref.mean():.2f}m")

REFERENCE set (n=6):
  truth=(39.952309,-75.191574)  pred=(39.951492,-75.191345)  haversine=92.85m
  truth=(39.952324,-75.191582)  pred=(39.951550,-75.191429)  haversine=87.08m
  truth=(39.952301,-75.191551)  pred=(39.951538,-75.191399)  haversine=85.83m
  truth=(39.952301,-75.191551)  pred=(39.951546,-75.191422)  haversine=84.71m
  truth=(39.952309,-75.191574)  pred=(39.951599,-75.191429)  haversine=79.86m
  truth=(39.952301,-75.191551)  pred=(39.951595,-75.191483)  haversine=78.69m
reference mean Haversine: 84.84m
